# Modèle de prédiction du risque de diabète

Notebook d'entraînement — équipe Data Science.

Objectif : à partir d'un profil patient (grossesses, glycémie, tension, IMC, etc.), prédire la présence d'un risque de diabète.

**Statut : exploration terminée, modèle validé. Rien n'est packagé ni déployé, et aucun suivi de performance n'existe — c'est l'objet du TP Module 5.**

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import joblib

df = pd.read_csv("../data/raw/diabetes_train.csv")
df.head()

## 1. Description des données

| Colonne | Description |
|---|---|
| pregnancies | Nombre de grossesses |
| glucose | Glycémie plasmatique (test de tolérance au glucose) |
| blood_pressure | Tension artérielle diastolique (mm Hg) |
| skin_thickness | Épaisseur du pli cutané tricipital (mm) |
| insulin | Insuline sérique à 2h (mu U/ml) |
| bmi | Indice de masse corporelle |
| diabetes_pedigree | Fonction pedigree du diabète (facteur héréditaire) |
| age | Âge du patient |
| outcome | 1 = risque de diabète présent, 0 = absent |

In [ ]:
df.describe()

In [ ]:
df["outcome"].value_counts(normalize=True)

~30% de cas positifs. Pas de valeurs manquantes sur ce jeu.

## 2. Séparation train / test

In [ ]:
FEATURES = [
    "pregnancies", "glucose", "blood_pressure", "skin_thickness",
    "insulin", "bmi", "diabetes_pedigree", "age",
]

X = df[FEATURES]
y = df["outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

## 3. Entraînement

RandomForest avec `class_weight="balanced"` : le déséquilibre des classes (~30% de positifs) pénalisait fortement le rappel sans cette pondération, or c'est le critère prioritaire pour un usage médical (un faux négatif est plus coûteux qu'un faux positif).

In [ ]:
model = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=3,
    class_weight="balanced", random_state=42,
)
model.fit(X_train, y_train)

## 4. Évaluation

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("recall:  ", round(recall_score(y_test, y_pred), 4))
print("f1:      ", round(f1_score(y_test, y_pred), 4))
print("auc:     ", round(roc_auc_score(y_test, y_proba), 4))
print("confusion matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
importances

Les variables les plus discriminantes sont cohérentes avec la littérature clinique : `glucose`, `bmi`, `age`, `diabetes_pedigree`.

## 5. Sauvegarde du modèle

Le modèle validé est sauvegardé en `.pkl`. **À partir d'ici, ce n'est plus le travail de la data scientist : le suivi en production (MLflow, monitoring, détection de dérive) et l'intégration continue (CI/CD) sont à la charge de l'équipe MLOps (TP Module 5).**

In [ ]:
joblib.dump({"model": model, "features": FEATURES}, "../diabetes_risk_model.pkl")